In [1]:
import sys
sys.path.append("/Users/monikagadage/Documents/java-interview-coach")

In [2]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages

class InterviewState(TypedDict):
    topic: str
    current_question: str
    user_answer: str
    feedback: str
    score: int
    total_questions: int
    needs_hint: bool
    hint: str
    weak_topics: list[str]
    next_action: str
    messages: Annotated[list, add_messages]

In [3]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END


In [4]:
llm = ChatGroq(model="llama-3.3-70b-versatile")

In [5]:
question_prompt = ChatPromptTemplate.from_template("""
You are a Java technical interviewer.
Generate ONE short, simple interview question about: {topic}
The question should be direct and answerable in 1-2 sentences.
No multi-part questions. No "explain and provide an example" style.
Just one clean, focused question.
""")

eval_prompt = ChatPromptTemplate.from_template("""
You are a Java technical interviewer evaluating an answer.

Question: {question}
Candidate's Answer: {answer}

Respond with:
1. CORRECT or INCORRECT
2. Brief feedback (2-3 sentences)
3. Ideal answer in simple terms

Start your response with either CORRECT or INCORRECT on the first line.
""")

hint_prompt = ChatPromptTemplate.from_template("""
You are a helpful Java tutor.
Give a short hint (2-3 sentences) for this question without giving away the answer.
Question: {question}
""")


In [6]:
question_chain = question_prompt | llm
eval_chain = eval_prompt | llm
hint_chain = hint_prompt | llm



In [7]:
def ask_question_node(state : InterviewState) -> dict:
    """Generates a new Interview Question"""
    response = question_chain.invoke({"topic" : state["topic"]})
    return{
        "current_question" : response.content,
        "total_questions" : state.get("total_questions", 0) + 1,
        "need_hints" : False,
        "hint" : ""
    }
    

In [8]:
def evaluate_node(state: InterviewState) -> dict:
    """Evaluates the user's answers"""
    response = eval_chain.invoke({
        "question": state["current_question"],
        "answer" : state["user_answer"]
    })
    
    feedback = response.content
    is_correct = feedback.strip().upper().startswith("CORRECT")
    
    score = state.get("score", 0)
    weak_topics = state.get("weak_topics" , [])
    
    if is_correct:
        score+=1
    else:
        weak_topics = weak_topics+ [state["topic"]]
        
    return {
        "feedback" : feedback,
        "score" : score,
        "weak_topics" : weak_topics
    }
    

In [9]:
def hint_node(state: InterviewState) -> dict:
    """Gives a hint for the current question"""
    response = hint_chain.invoke({
        "question": state["current_question"]
    })
    return {"hint": response.content}

In [10]:
def end_node(state: InterviewState) -> dict:
    """Shows final score"""
    return {"next_action": "end"}

In [11]:
def router(state: InterviewState) -> str:
    """Decides what to do next based on next_action"""
    action = state.get("next_action", "ask")

    if action == "end":
        return "end"
    elif action == "hint":
        return "hint"
    elif action == "evaluate":
        return "evaluate"
    else:
        return "ask"

In [12]:
def build_graph():
    graph = StateGraph(InterviewState)
    
    graph.add_node("ask" , ask_question_node)
    graph.add_node("evaluate", evaluate_node)
    graph.add_node("hint", hint_node)
    graph.add_node("end", end_node)
    
    graph.set_entry_point("ask")
    
    graph.add_edge("ask" , "evaluate")
    
    graph.add_conditional_edges(
        "evaluate",
        router,{
            "ask": "ask",
            "hint": "hint",
            "end": END
        }
    )
    graph.add_edge("hint", "evaluate")

    return graph.compile()


In [13]:
def run_interview():
    print("🎯 Java Interview Coach\n")
    sys.stdout.flush()
    
    topic = input("Enter a topic: ")
    print(f"\n⏳ Generating question on {topic}...\n")
    sys.stdout.flush()

    state = {
        "topic": topic,
        "current_question": "",
        "user_answer": "",
        "feedback": "",
        "score": 0,
        "total_questions": 0,
        "needs_hint": False,
        "hint": "",
        "weak_topics": [],
        "next_action": "ask",
        "messages": []
    }

    for _ in range(5):
        # Step 1 — generate question directly
        result = ask_question_node(state)
        state.update(result)
        print(f"\n📌 Question {state['total_questions']}:\n{state['current_question']}\n")
        sys.stdout.flush()

        # Step 2 — get answer
        user_input = input("Your answer (or type 'hint'): ")

        # Step 3 — hint if needed
        if user_input.lower() == "hint":
            hint_result = hint_node(state)
            state.update(hint_result)
            print(f"\n💡 Hint:\n{state['hint']}\n")
            sys.stdout.flush()
            user_input = input("Now your answer: ")

        # Step 4 — evaluate
        state["user_answer"] = user_input
        eval_result = evaluate_node(state)
        state.update(eval_result)
        print(f"\n💬 Feedback:\n{state['feedback']}\n")
        print(f"⭐ Score: {state['score']}/{state['total_questions']}")
        sys.stdout.flush()

        cont = input("\nNext question? (y/n): ")
        if cont.lower() != "y":
            break

        state["next_action"] = "ask"

    print(f"\n🏁 Session Complete!")
    print(f"✅ Final Score: {state['score']}/{state['total_questions']}")
    if state["weak_topics"]:
        print(f"📚 Topics to review: {', '.join(set(state['weak_topics']))}")

run_interview()

🎯 Java Interview Coach


⏳ Generating question on ...


📌 Question 1:
What is the primary difference between the '==' operator and the 'equals()' method in Java?


💬 Feedback:
INCORRECT
The candidate's answer, "OOP", does not address the question about the difference between the '==' operator and the 'equals()' method in Java. It seems like the candidate may have misunderstood the question or provided an unrelated term. 

An ideal answer in simple terms would be: The '==' operator checks if both objects point to the same memory location, whereas the 'equals()' method checks if the contents of the two objects are equal, regardless of their memory locations.

⭐ Score: 0/1

🏁 Session Complete!
✅ Final Score: 0/1
📚 Topics to review: 
